## RDKit Fingerprints

https://www.rdkit.org/docs/GettingStartedInPython.html

In [1]:
import pickle
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit import DataStructs
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd


MORGAN_FP = True

# by default drugs_df contains all drugs that have ever been tried against the
# group of diseases of interest
# although only safe ones are considered for further analysis
# - one can use all or only safe drugs to get PCs of molecular fingerprints
# data relevance vs data quantity
USE_ONLY_SAFE_DRUGS = True 

In [2]:
with open("../data/01-result/drugs_df.pkl","rb") as f:
    drugs_df =pickle.load(f)

if USE_ONLY_SAFE_DRUGS:
    drugs_df = drugs_df[ (drugs_df["max_phase"] == "4.0") | (drugs_df["max_phase"] == "3.0")]

In [3]:
drugs_df = drugs_df[ drugs_df["canonical_smiles"].notna() ]

smiles_list = drugs_df["canonical_smiles"]#["O=C(NCc1cc(OC)c(O)cc1)CCCC/C=C/C(C)C", "CC(C)CCCCCC(=O)NCC1=CC(=C(C=C1)O)OC", "c1(C=O)cc(OC)c(O)cc1"]

mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]

if MORGAN_FP:
    fpgen = AllChem.GetMorganGenerator(radius=2)
    fps = np.array([np.fromiter(fpgen.GetFingerprint(mol).ToBitString(), dtype=int)  for mol in mols])
else:
    fps = np.array([np.fromiter(Chem.RDKFingerprint(mol).ToBitString(), dtype=int)  for mol in mols])

In [4]:
pca = PCA(n_components=100)
X_reduced = pca.fit_transform(fps)
fp_df = pd.DataFrame(X_reduced, columns=[f'FP_{i+1}' for i in range(X_reduced.shape[1])])

In [5]:
fingerprints_df = pd.concat([drugs_df["drug_id"].reset_index(drop=True), fp_df], axis=1)
with open("../data/01-result/fingerprints_df.pkl","wb") as f:
    pickle.dump(fingerprints_df,f)